# D3 — DistilGPT-2 / SST-2 (5 + 5 epochs)
Phase 1: 5 epochs at 1e-4. Phase 2: 5 private Ghost Clipping epochs at 1e-4, target ε=3, No-WT. Results are isolated from GPT-2.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
!python3.12 --version
!apt-get update -qq
!apt-get install -y -qq python3.12-venv python3.12-dev
import os
if not os.path.exists('/content/ghostdp_env/bin/python'):
    !python3.12 -m venv /content/ghostdp_env


Python 3.12.3
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package python3-pip-whl.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../python3-pip-whl_24.0+dfsg-1ubuntu1.3_all.deb ...
Unpacking python3-pip-whl (24.0+dfsg-1ubuntu1.3) ...
Selecting previously unselected package python3-setuptools-whl.
Preparing to unpack .../python3-setuptools-whl_68.1.2-2ubuntu1.2_all.deb ...
Unpacking python3-setuptools-whl (68.1.2-2ubuntu1.2) ...
Selecting previously unselected package python3.12-dev.
Preparing to unpack .../python3.12-dev_3.12.3-1ubuntu0.17_amd64.deb ...
Unpacking python3.12-dev (3.12.3-1ubuntu0.17) ...
Selecting previously unselected package python3.12-venv.
Preparing to unpack .../python3.12-venv_3.12.3-1ubuntu0.17_amd64.deb ...
Unpacking python3.12-venv (3.12.3-1

In [4]:
!/content/ghostdp_env/bin/python -m pip install -q 'torch==2.6.0' 'torchvision==0.21.0' 'torchaudio==2.6.0' --index-url https://download.pytorch.org/whl/cu124


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 191.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 234.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 144.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 92.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 118.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 159.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 135.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 129.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 131.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 131.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 112.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!/content/ghostdp_env/bin/python -m pip install -q 'transformers==4.44.2' 'tokenizers==0.19.1' 'huggingface_hub==0.24.7' 'datasets==2.21.0' 'numpy==1.26.4' 'scipy==1.12.0' 'psutil' 'tqdm' 'opt_einsum' 'prv-accountant'
!/content/ghostdp_env/bin/python -m pip install -q --no-deps git+https://github.com/lxuechen/private-transformers.git
!/content/ghostdp_env/bin/python -m pip install -q --no-deps git+https://github.com/lxuechen/ml-swissknife.git


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
!/content/ghostdp_env/bin/python -m pip install -q 'spacy==3.7.5'
!/content/ghostdp_env/bin/python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.8/260.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/134.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.1/487.1 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.3/123.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
import subprocess
# Install missing dependency 'click' and downgrade pyarrow to make it compatible with NumPy 1.x
subprocess.run(['/content/ghostdp_env/bin/python', '-m', 'pip', 'install', '-q', 'click', 'pyarrow<15.0.0'], check=True)

code='''import torch, transformers, datasets, numpy, scipy, spacy\nfrom private_transformers import PrivacyEngine\nprint("GPU:",torch.cuda.get_device_name(0))\nprint("Transformers:",transformers.__version__)\nprint("PrivacyEngine: OK")''' if 'D3' == 'D3' else '''import torch, transformers, datasets, numpy, scipy\nfrom private_transformers import PrivacyEngine\nprint("GPU:",torch.cuda.get_device_name(0))\nprint("Transformers:",transformers.__version__)\nprint("PrivacyEngine: OK")'''

try:
    subprocess.run(['/content/ghostdp_env/bin/python', '-c', code], check=True, capture_output=True, text=True)
    print("Subprocess completed successfully.")
except subprocess.CalledProcessError as e:
    print("Subprocess failed with the following output:")
    print("STDOUT:", e.stdout)
    print("STDERR:", e.stderr)
    raise e

Subprocess completed successfully.


## Seeds and output directory
Change `SEEDS` below as needed. Each seed writes to its own Google Drive folder. Avoid running multiple notebooks simultaneously on the same GPU.


In [10]:
SEEDS = [40, 50, 60, 6, 37]
OUTPUT_DIR = '/content/drive/MyDrive/DualTune_GhostDP/DistilGPT2/multiseed'


In [11]:
from pathlib import Path
script_path=Path('/content/D3_DistilGPT2_SST2_multiseed.py')
script_path.write_text('#!/usr/bin/env python3\n"""DistilGPT-2 SST-2 reproducible experiment. Run under the Python 3.12 Colab venv."""\nimport argparse, os, time, random, json, csv, gc\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\nfrom transformers import GPT2TokenizerFast, GPT2LMHeadModel\nfrom datasets import load_dataset, Dataset\nfrom private_transformers import PrivacyEngine\nfrom tqdm.auto import tqdm\nimport psutil\nfrom datetime import datetime\n\nLOG_FILE = None\n\ndef log(message):\n    stamp = datetime.now().strftime(\'%Y-%m-%d %H:%M:%S\')\n    line = f\'[{stamp}] {message}\'\n    print(line, flush=True)\n    if LOG_FILE is not None:\n        with open(LOG_FILE, \'a\', encoding=\'utf-8\') as f:\n            f.write(line + \'\\n\')\n\ndef progress(seed, phase, epoch, step, total, loss, started):\n    done = step + 1\n    elapsed = time.monotonic() - started\n    eta = elapsed / done * (total - done)\n    mem = torch.cuda.memory_allocated() / 2**20\n    log(f\'Seed {seed} | {phase} | Epoch {epoch}/{PHASE1_EPOCHS if phase == \'Phase 1\' else DP_EPOCHS} | \'\n        f\'{done}/{total} batches ({done/total:.1%}) | \'\n        f\'loss={loss:.4f} | elapsed={elapsed/60:.1f} min | \'\n        f\'ETA={eta/60:.1f} min | GPU allocated={mem:.0f} MB\')\n\nMODEL_NAME=\'distilgpt2\'; MAX_LEN=128; BATCH_SIZE=24\nPHASE1_EPOCHS=5; DP_EPOCHS=5; TARGET_EPSILON=3.0\nMAX_GRAD_NORM=0.05; LR_P1=1e-4; LR_DP=1e-4\nPOS_STR=\' positive\'; NEG_STR=\' negative\'\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic=True\n    torch.backends.cudnn.benchmark=False\n\ndef balance(ds,seed):\n    # Exact Python random.sample sequence from original uploaded notebook.\n    a=[ex for ex in ds if ex[\'label\']==0]\n    b=[ex for ex in ds if ex[\'label\']==1]\n    random.seed(seed)\n    n=min(len(a),len(b))\n    chosen=random.sample(a,n)+random.sample(b,n)\n    random.shuffle(chosen)\n    return Dataset.from_list(chosen)\n\ndef collate(batch):\n    return {k:torch.tensor([x[k] for x in batch],dtype=torch.long)\n            for k in (\'input_ids\',\'attention_mask\',\'label_token\')}\n\ndef last_logits(model,ids,mask):\n    logits=model(input_ids=ids,attention_mask=mask).logits\n    return logits[torch.arange(ids.shape[0],device=ids.device),\n                  (mask.sum(1)-1).clamp(min=0),:]\n\ndef eval_acc(model,loader,device,pos,neg):\n    model.eval(); good=total=0\n    with torch.no_grad():\n        for batch in loader:\n            ids=batch[\'input_ids\'].to(device); mask=batch[\'attention_mask\'].to(device)\n            target=batch[\'label_token\'].to(device)\n            logits=last_logits(model,ids,mask)\n            pred=torch.where(logits[:,pos]>=logits[:,neg],pos,neg)\n            good+=(pred==target).sum().item(); total+=len(target)\n    model.train()\n    return good/total\n\ndef setup_data(seed):\n    raw=load_dataset(\'nyu-mll/glue\',\'sst2\')\n    train=balance(raw[\'train\'],seed); val=balance(raw[\'validation\'],seed)\n    tok=GPT2TokenizerFast.from_pretrained(MODEL_NAME)\n    tok.pad_token=tok.eos_token\n    pos=tok.encode(POS_STR,add_special_tokens=False)\n    neg=tok.encode(NEG_STR,add_special_tokens=False)\n    assert len(pos)==len(neg)==1\n    pos,neg=pos[0],neg[0]\n    def encode(ex):\n        encoded=tok(f"Review: {ex[\'sentence\']}\\nSentiment:",padding=\'max_length\',\n                    truncation=True,max_length=MAX_LEN)\n        return {\'input_ids\':encoded[\'input_ids\'],\n                \'attention_mask\':encoded[\'attention_mask\'],\n                \'label_token\':pos if ex[\'label\']==1 else neg}\n    val_tok=val.map(encode)\n    val_loader=DataLoader(val_tok,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate)\n    return train,encode,val_loader,tok,pos,neg\n\ndef build_model(tok,weight_tying,device):\n    m=GPT2LMHeadModel.from_pretrained(MODEL_NAME)\n    m.config.pad_token_id=tok.pad_token_id\n    if weight_tying:\n        m.tie_weights()\n    else:\n        m.lm_head.weight=nn.Parameter(m.lm_head.weight.detach().clone())\n    m.to(device)\n    tied=(m.lm_head.weight.data_ptr()==m.transformer.wte.weight.data_ptr())\n    assert tied==weight_tying,f\'WT check failed: {tied}\'\n    return m\n\ndef resource_start():\n    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()\n    return time.time(),psutil.Process().memory_info().rss/2**20\n\ndef resource_end(start):\n    t,rss=start\n    return dict(time_min=(time.time()-t)/60,\n                gpu_peak_allocated_mb=torch.cuda.max_memory_allocated()/2**20,\n                gpu_peak_reserved_mb=torch.cuda.max_memory_reserved()/2**20,\n                cpu_rss_delta_mb=psutil.Process().memory_info().rss/2**20-rss)\n\ndef train_dp(model,train_tok,val_loader,device,pos,neg,clipping,seed):\n    loader=DataLoader(train_tok,batch_size=BATCH_SIZE,shuffle=True,drop_last=True,collate_fn=collate)\n    opt=torch.optim.Adam(model.parameters(),lr=LR_DP)\n    pe=PrivacyEngine(model,batch_size=BATCH_SIZE,sample_size=len(train_tok),\n                     epochs=DP_EPOCHS,max_grad_norm=MAX_GRAD_NORM,\n                     target_epsilon=TARGET_EPSILON,clipping_mode=clipping)\n    pe.attach(opt)\n    log(f"Seed {seed} | Phase 2 | PrivacyEngine attached | {len(loader)} batches/epoch")\n    results=[]\n    for epoch in range(DP_EPOCHS):\n        model.train(); total_loss=0\n        epoch_start = time.monotonic()\n        log(f\'Seed {seed} | Phase 2 | START epoch {epoch+1}/{DP_EPOCHS}\')\n        for step,batch in enumerate(loader):\n            ids=batch[\'input_ids\'].to(device); mask=batch[\'attention_mask\'].to(device)\n            target=batch[\'label_token\'].to(device)\n            opt.zero_grad()\n            loss=F.cross_entropy(last_logits(model,ids,mask),target,reduction=\'none\')\n            opt.step(loss=loss)\n            total_loss+=loss.detach().mean().item()\n            if step==0 or (step+1)%100==0 or step+1==len(loader):\n                progress(seed,\'Phase 2\',epoch+1,step,len(loader),loss.mean().item(),epoch_start)\n        acc=eval_acc(model,val_loader,device,pos,neg)\n        log(f\'Seed {seed} | Phase 2 | END epoch {epoch+1} | mean_loss={total_loss/len(loader):.4f} | val_acc={acc:.4f} | elapsed={(time.monotonic()-epoch_start)/60:.1f} min\')\n        results.append({\'epoch\':epoch+1,\'loss\':total_loss/len(loader),\'validation_accuracy\':acc})\n    # Do not pass epoch count as `steps`; use actual optimizer steps for privacy accounting.\n    spent=None\n    try:\n        spent=pe.get_privacy_spent(steps=DP_EPOCHS*len(loader))\n    except Exception as e:\n        raise RuntimeError(\'Privacy accountant query failed; do not report this seed as privacy-verified\') from e\n    if spent is None:\n        raise RuntimeError(\'Privacy accountant returned no result\')\n    return results,repr(spent)\n\ndef save_result(args,config,seed,model,metrics,epochs,privacy):\n    out=Path(args.output_dir)/config/f\'seed_{seed}\';out.mkdir(parents=True,exist_ok=True)\n    torch.save(model.state_dict(),out/\'final_model.pth\')\n    record={\'configuration\':config,\'seed\':seed,\'gpu\':torch.cuda.get_device_name(0),\n            \'target_epsilon\':TARGET_EPSILON,\'max_grad_norm\':MAX_GRAD_NORM,\n            \'batch_size\':BATCH_SIZE,\'dp_epochs\':DP_EPOCHS,\'privacy_report\':privacy,\n            \'validation_accuracy\':epochs[-1][\'validation_accuracy\'],**metrics,\n            \'epoch_metrics\':epochs}\n    (out/\'results.json\').write_text(json.dumps(record,indent=2))\n    with (Path(args.output_dir)/f\'{config}_summary.csv\').open(\'a\',newline=\'\') as f:\n        row={k:v for k,v in record.items() if k!=\'epoch_metrics\'}\n        writer=csv.DictWriter(f,fieldnames=list(row))\n        if f.tell()==0:writer.writeheader()\n        writer.writerow(row)\n    print(\'SAVED\',out,flush=True)\n\ndef parse():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--seeds\',nargs=\'+\',type=int,default=[60])\n    p.add_argument(\'--output-dir\',default=\'/content/drive/MyDrive/DualTune_GhostDP/multiseed\')\n    return p.parse_args()\n\ndef sanitized_training(train,encode,tok,device,seed,save_path):\n    import spacy\n    log(f\'Seed {seed} | Phase 1 | Loading spaCy\')\n    nlp=spacy.load(\'en_core_web_sm\')\n    log(f\'Seed {seed} | Phase 1 | Sanitizing {len(train)} examples\')\n    sanitize_start=time.monotonic()\n    def sanitize(text):\n        doc=nlp(text); redacted=[]\n        for token in doc:\n            if (token.ent_type_ or token.pos_ in {\'PRON\',\'PROPN\',\'VERB\'}\n                    or token.dep_ in {\'nsubj\',\'dobj\',\'pobj\'}):\n                redacted.append(f\'<{token.pos_}>\')\n            else:redacted.append(token.text)\n        return \' \'.join(redacted)\n    sanitized=train.map(lambda ex:{\'sentence\':sanitize(ex[\'sentence\'])})\n    log(f\'Seed {seed} | Phase 1 | Sanitization complete in {(time.monotonic()-sanitize_start)/60:.1f} min; tokenizing\')\n    train_tok=sanitized.map(encode)\n    loader=DataLoader(train_tok,batch_size=BATCH_SIZE,shuffle=True,drop_last=True,collate_fn=collate)\n    model=build_model(tok,False,device)\n    opt=torch.optim.AdamW(model.parameters(),lr=LR_P1)\n    log(f\'Seed {seed} | Phase 1 | Training {len(loader)} batches/epoch\')\n    for epoch in range(PHASE1_EPOCHS):\n        model.train(); running=0\n        epoch_start=time.monotonic()\n        log(f\'Seed {seed} | Phase 1 | START epoch {epoch+1}/{PHASE1_EPOCHS}\')\n        for step,batch in enumerate(loader):\n            ids=batch[\'input_ids\'].to(device);mask=batch[\'attention_mask\'].to(device)\n            target=batch[\'label_token\'].to(device)\n            opt.zero_grad()\n            loss=F.cross_entropy(last_logits(model,ids,mask),target,reduction=\'mean\')\n            loss.backward();opt.step();running+=loss.item()\n            if step==0 or (step+1)%100==0 or step+1==len(loader):\n                progress(seed,\'Phase 1\',epoch+1,step,len(loader),loss.item(),epoch_start)\n        log(f\'Seed {seed} | Phase 1 | END epoch {epoch+1} | mean_loss={running/len(loader):.4f} | elapsed={(time.monotonic()-epoch_start)/60:.1f} min\')\n    log(f\'Seed {seed} | Phase 1 | Saving checkpoint to {save_path}\')\n    torch.save(model.state_dict(),save_path)\n    log(f\'Seed {seed} | Phase 1 | Checkpoint saved\')\n    del model,opt,loader;gc.collect();torch.cuda.empty_cache()\n\n\ndef main():\n    args=parse();Path(args.output_dir).mkdir(parents=True,exist_ok=True)\n    assert torch.cuda.is_available(),\'Colab GPU required\'\n    device=torch.device(\'cuda\')\n    global LOG_FILE\n    LOG_FILE=Path(args.output_dir)/\'D3_progress.log\'\n    log(f\'CONFIG D3 | WT=False | clipping=ghost | GPU={torch.cuda.get_device_name(0)} | seeds={args.seeds}\')\n    for seed in args.seeds:\n        log(f\'===== START SEED {seed} =====\')\n        seed_start=time.monotonic()\n        seed_all(seed)\n        log(f\'Seed {seed} | Loading and balancing SST-2\')\n        train,encode,val_loader,tok,pos,neg=setup_data(seed)\n        log(f\'Seed {seed} | Dataset ready: {len(train)} training examples\')\n        out=Path(args.output_dir)/\'D3\'/f\'seed_{seed}\'\n        out.mkdir(parents=True,exist_ok=True)\n        checkpoint=out/\'phase1_sanitized.pth\'\n        metadata_path=out/\'phase1_config.json\'\n        expected={\'model\':MODEL_NAME,\'seed\':seed,\'phase1_epochs\':PHASE1_EPOCHS,\'phase1_lr\':LR_P1,\'weight_tying\':False}\n        if checkpoint.exists():\n            if not metadata_path.exists() or json.loads(metadata_path.read_text()) != expected:\n                raise RuntimeError(f\'Existing checkpoint at {checkpoint} has missing/mismatched configuration metadata; use a fresh output directory\')\n        else:\n            sanitized_training(train,encode,tok,device,seed,checkpoint)\n            metadata_path.write_text(json.dumps(expected,indent=2))\n        if metadata_path.exists():\n            log(f\'Seed {seed} | Phase 1 checkpoint ready\')\n        # Phase-2 RNG and model construction follow original script\'s fresh-model pattern.\n        log(f\'Seed {seed} | Phase 2 | Loading Phase 1 checkpoint\')\n        model=build_model(tok,False,device)\n        state=torch.load(checkpoint,map_location=\'cpu\',weights_only=True)\n        model.load_state_dict(state,strict=True);del state\n        assert model.lm_head.weight.data_ptr()!=model.transformer.wte.weight.data_ptr()\n        train_tok=train.map(encode)\n        start=resource_start()\n        epochs,privacy=train_dp(model,train_tok,val_loader,device,pos,neg,\'ghost\',seed)\n        metrics=resource_end(start)\n        log(f\'Seed {seed} | Saving final model and results\')\n        save_result(args,\'D3\',seed,model,metrics,epochs,privacy)\n        log(f\'Seed {seed} | COMPLETE | total elapsed={(time.monotonic()-seed_start)/60:.1f} min\')\n        del model,train_tok,train,val_loader,tok;gc.collect();torch.cuda.empty_cache()\n\nif __name__==\'__main__\':main()\n')
print('Wrote',script_path)


Wrote /content/D3_DistilGPT2_SST2_multiseed.py


In [12]:
import subprocess
# Download the missing spaCy pipeline model in the environment before starting
subprocess.run(['/content/ghostdp_env/bin/python', '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

cmd=['/content/ghostdp_env/bin/python','-u',str(script_path),'--seeds',*map(str,SEEDS),'--output-dir',OUTPUT_DIR]
print('Starting:', ' '.join(cmd), flush=True)
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc=proc.wait()
if rc: raise RuntimeError(f'Experiment failed with exit code {rc}')

Streaming output truncated to the last 5000 lines.
Map: 100%|██████████| 59560/59560 [06:04<00:00, 163.39 examples/s]
[2026-10-09 14:06:24] Seed 6 | Phase 1 | Sanitization complete in 6.1 min; tokenizing

Map: 100%|██████████| 59560/59560 [00:17<00:00, 3376.70 examples/s]
[2026-10-09 14:06:42] Seed 6 | Phase 1 | Training 2481 batches/epoch
[2026-10-09 14:06:42] Seed 6 | Phase 1 | START epoch 1/5
[2026-10-09 14:06:42] Seed 6 | Phase 1 | Epoch 1/5 | 1/2481 batches (0.0%) | loss=8.2512 | elapsed=0.0 min | ETA=5.9 min | GPU allocated=1868 MB
[2026-10-09 14:06:53] Seed 6 | Phase 1 | Epoch 1/5 | 100/2481 batches (4.0%) | loss=0.3321 | elapsed=0.2 min | ETA=4.4 min | GPU allocated=1869 MB
[2026-10-09 14:07:05] Seed 6 | Phase 1 | Epoch 1/5 | 200/2481 batches (8.1%) | loss=0.2707 | elapsed=0.4 min | ETA=4.2 min | GPU allocated=1869 MB
[2026-10-09 14:07:16] Seed 6 | Phase 1 | Epoch 1/5 | 300/2481 batches (12.1%) | loss=0.3629 | elapsed=0.6 min | ETA=4.1 min | GPU allocated=1869 MB
[2026-10-09 14

In [13]:
from pathlib import Path
import pandas as pd
p=Path(OUTPUT_DIR)/'D3_summary.csv'
print('Summary:',p,'Exists:',p.exists())
if p.exists():
    display(pd.read_csv(p))


Summary: /content/drive/MyDrive/DualTune_GhostDP/DistilGPT2/multiseed/D3_summary.csv Exists: True


,configuration,seed,gpu,target_epsilon,max_grad_norm,batch_size,dp_epochs,privacy_report,validation_accuracy,time_min,gpu_peak_allocated_mb,gpu_peak_reserved_mb,cpu_rss_delta_mb
0,D3,40,NVIDIA A100-SXM4-40GB,3.0,0.05,24,5,"{'eps_rdp': 2.9861763209369974, 'alpha_rdp': 4.6}",0.839953,43.045639,4154.38916,4514.0,34.382812
1,D3,50,NVIDIA A100-SXM4-40GB,3.0,0.05,24,5,"{'eps_rdp': 2.9861763209369974, 'alpha_rdp': 4.6}",0.832944,43.043938,4153.88916,4514.0,-4.359375
2,D3,60,NVIDIA A100-SXM4-40GB,3.0,0.05,24,5,"{'eps_rdp': 2.9861763209369974, 'alpha_rdp': 4.6}",0.829439,43.091550,4153.88916,4514.0,-4.886719
3,D3,6,NVIDIA A100-SXM4-40GB,3.0,0.05,24,5,"{'eps_rdp': 2.9861763209369974, 'alpha_rdp': 4.6}",0.841121,43.030517,4153.88916,4514.0,-2.902344
4,D3,37,NVIDIA A100-SXM4-40GB,3.0,0.05,24,5,"{'eps_rdp': 2.9861763209369974, 'alpha_rdp': 4.6}",0.836449,43.157509,4153.88916,4514.0,-4.082031


In [14]:
from pathlib import Path
log_path=Path(OUTPUT_DIR)/'D3_progress.log'
print('Log:',log_path)
if log_path.exists():
    print(''.join(log_path.read_text().splitlines(keepends=True)[-40:]))


Log: /content/drive/MyDrive/DualTune_GhostDP/DistilGPT2/multiseed/D3_progress.log
[2026-10-09 16:14:51] Seed 37 | Phase 2 | Epoch 4/5 | 1700/2481 batches (68.5%) | loss=0.0001 | elapsed=5.9 min | ETA=2.7 min | GPU allocated=1869 MB
[2026-10-09 16:15:12] Seed 37 | Phase 2 | Epoch 4/5 | 1800/2481 batches (72.6%) | loss=1.7299 | elapsed=6.2 min | ETA=2.4 min | GPU allocated=1869 MB
[2026-10-09 16:15:33] Seed 37 | Phase 2 | Epoch 4/5 | 1900/2481 batches (76.6%) | loss=0.8524 | elapsed=6.6 min | ETA=2.0 min | GPU allocated=1869 MB
[2026-10-09 16:15:53] Seed 37 | Phase 2 | Epoch 4/5 | 2000/2481 batches (80.6%) | loss=1.3400 | elapsed=6.9 min | ETA=1.7 min | GPU allocated=1869 MB
[2026-10-09 16:16:14] Seed 37 | Phase 2 | Epoch 4/5 | 2100/2481 batches (84.6%) | loss=1.1125 | elapsed=7.3 min | ETA=1.3 min | GPU allocated=1869 MB
[2026-10-09 16:16:35] Seed 37 | Phase 2 | Epoch 4/5 | 2200/2481 batches (88.7%) | loss=0.2113 | elapsed=7.6 min | ETA=1.0 min | GPU allocated=1869 MB
[2026-10-09 16:16: